In [ ]:
#import necessary package
import torch
import copy
import torch.nn.utils.prune as prune
from torchvision import transforms, datasets, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.quantization as tq
from tqdm.notebook import tqdm

import multiprocessing
import os

In [ ]:
# preprocess images
input_size = (224,224)
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(input_size),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)
# Load model
model = torch.load("fruit_mobilenetv2.pth", weights_only=False)

# Training SetUp
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.train()
# torch.backends.quantized.engine = "fbgemm"  # x86
# model = model.to(device)

In [ ]:
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

In [ ]:
def print_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    print("%.2f MB" %(os.path.getsize("tmp.pt")/1e6))
    os.remove('tmp.pt')

def evaluate(model, criterion, data_loader, device,epoch):
    
    model.eval()
    
    epoch_loss = 0.0
    
    correct_predictions = 0
    total_predictions = 0
    
    num_batches = len(data_loader)
    
    with torch.no_grad():
       
        for image, target in tqdm(data_loader):
            image, target = image.to(device), target.to(device)
            output = model(image)
            loss = criterion(output, target)
            # Accumulate batch loss
            epoch_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(output, 1)  # Get the predicted class index
            correct_predictions += (predicted == target).sum().item()
            total_predictions += target.size(0)
            
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    accuracy = correct_predictions / total_predictions
    
    print(f"Epoch = {epoch+1} || Test Loss: {avg_epoch_loss:.4f} || Test Accuracy: {accuracy:.4f}")

 #===================train function=========================

def train_epoch(model, criterion, optimizer, train_loader, device, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = len(train_loader)
    
    for batch_idx, (image, target) in enumerate(tqdm(train_loader)):
        image, target = image.to(device), target.to(device)
        output = model(image)
        loss = criterion(output, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Epoch = {num_epochs+1} || Training Loss: {avg_epoch_loss:.4f}")

In [ ]:
print_model_size(model)

In [ ]:
import platform
chip = platform.processor()

if chip == 'arm':
    backend = 'qnnpack'
elif chip in ['x86_64', 'i386']:
    backend = 'fbgemm'
else:
    raise SystemError("Backend is not supported")

print(f"Using {backend} backend engine for {chip} CPU")


torch.backends.quantized.engine = backend

In [ ]:
from torch.quantization.quantize_fx import prepare_fx, convert_fx,prepare_qat_fx

example_inputs = (torch.randn(1, 3, 224, 224),)
qconfig = {
    "": torch.quantization.get_default_qat_qconfig(backend),
    "module_name": {
      #  "features.1.conv.1", None,    
      #  "features.2.conv.0.0", None,
      #  "features.2.conv.2", None,
      #  "features.3.conv.1.0", None,
      #  "features.3.conv.2", None,
      #  "features.4.conv.1.0", None,
      #  "features.4.conv.2", None,
      #  "features.5.conv.1.0", None,
      #  "features.7.conv.1.0", None,
      #  "features.8.conv.1.0", None,
      #  "features.9.conv.2", None,
      #  "features.11.conv.1.0", None,
      #  "features.12.conv.2", None,
      #  "features.14.conv.1.0", None,
      #  "features.14.conv.2", None,
      #  "features.15.conv.1.0", None,
      #  "features.16.conv.2", None,
      #  "features.17.conv.0.0", None,
      #  "features.17.conv.2", None,

    }
}
prepared_model = prepare_qat_fx(model.train(), qconfig, example_inputs)

In [ ]:
# Train with QAT
num_epochs = 10
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.AdamW(prepared_model.parameters(), lr = 0.0001)
for nepoch in range(num_epochs):
    train_epoch(prepared_model, criterion, optimizer, train_loader, device, nepoch)
    print("Evaluating quantized model...")
    model_quantized = copy.deepcopy(prepared_model)
    model_quantized.to(torch.device("cpu"))
    model_quantized = convert_fx(model_quantized.eval())
    evaluate(model_quantized,criterion, test_loader,torch.device("cpu"),nepoch)

    # Save the quantized model as a scripted fx model
    model_quantized.eval()
    scripted_model = torch.jit.trace(model_quantized, example_inputs)
    scripted_model.save("../model/Scriptedfx_int8_fruit_mobilenetv2.pt")